# Démonstration d'inférence U-TILISE

Ce notebook montre comment :
1. Charger un modèle entraîné et un dataset de test
2. Comparer les 3 stratégies de masquage (`random_clouds`, `random_fully_masked`, `consecutive_fully_masked`)
   en utilisant les paramètres `t_sampled` et `t_masked` de `SentinelDataset.__getitem__`
3. Produire et visualiser les reconstructions
4. Calculer les métriques de reconstruction

> **Prérequis :** environnement conda `cloud_reconstruction` activé, checkpoint et HDF5 accessibles.

In [7]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()

from src import config_utils, data_utils
from src.eval_tools import Imputation
from src.data_utils import extract_sample

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {'cuda' if torch.cuda.is_available() else 'cpu'}")

PyTorch : 2.6.0+cu124
Device  : cuda


## 1. Chargement du dataset et du modèle

In [8]:
CHECKPOINT = os.environ["CHECKPOINT"]
CONFIG_TRAIN = os.environ["TRAIN_CONFIG"]
HDF5_FILE = os.environ["HDF5_FILE"]

TEMPORAL_WINDOW = 14
BLEND_MODE = "center"

print(f"Checkpoint : {CHECKPOINT}")
print(f"Config     : {CONFIG_TRAIN}")
print(f"HDF5       : {HDF5_FILE}")

# Charger la config et le dataset de test
config = config_utils.read_config_with_defaults(
    os.path.join(PROJECT_ROOT, "configs/config_run_eval.yaml"),
    run_mode="test",
)
config.data.hdf5_file = HDF5_FILE

test_dset = data_utils.get_dataset(config, phase="test")
print(f"\nDataset : {len(test_dset)} echantillons")
print(f"Canaux  : {test_dset.num_channels}")

Checkpoint : /mnt/stores/store_dai/tmp/speillet/cloud_reconstruction_results/U-TILISE/v4_asc_desc_random_clouds_combined/2026-03-29_16-10/checkpoints/Model_best.pth
Config     : /mnt/stores/store_dai/tmp/speillet/cloud_reconstruction_results/U-TILISE/v4_asc_desc_random_clouds_combined/2026-03-29_16-10/config.yaml
HDF5       : /mnt/DATA_10T/data_rpg/circa/hdf5/CIRCA_CR_merged.hdf5


FileNotFoundError: ERROR: Cannot find the file /home/SPeillet/rpg_3STR/cloud_reconstruction/configs/config_run_eval.yaml


In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
imputer = Imputation(
    config_file_train=CONFIG_TRAIN,
    checkpoint=CHECKPOINT,
    temporal_window=TEMPORAL_WINDOW,
    blend_mode=BLEND_MODE,
    num_channels=test_dset.num_channels,
    device=device,
)
print(f"Modele charge sur {device}")

Checkpoint '/mnt/stores/store_dai/tmp/speillet/cloud_reconstruction_results/U-TILISE/v4_asc_desc_random_clouds_combined/2026-03-29_16-10/checkpoints/Model_best.pth' loaded.
Chosen epoch: 110

Modele charge sur cuda:0


## 2. Charger un échantillon brut

On charge l'échantillon depuis le backend pour connaître la taille de la série temporelle,
puis on fixe `t_sampled` (fenêtre temporelle) et on définit les 3 stratégies de `t_masked`.

In [ ]:
SAMPLE_IDX = 0

raw = test_dset.backend[SAMPLE_IDX]
T_total = len(raw["valid_obs"])
T = min(T_total, TEMPORAL_WINDOW)
t_sampled = torch.arange(0, T)

print(f"Echantillon #{SAMPLE_IDX}")
print(f"  Dates totales (valid_obs) : {T_total}")
print(f"  Fenetre retenue           : {T} dates (t0..t{T-1})")
print(f"  Dates S2                  : {len(raw['S2']['S2_dates'])}")

Echantillon #0
  Dates totales (valid_obs) : 30
  Fenetre retenue           : 14 dates (t0..t13)
  Dates S2                  : 75


## 3. Définition des 3 stratégies de masquage

`t_masked` est un dictionnaire avec deux clés :
- `indices_masked` : indices des dates (partiellement ou totalement) masquées
- `indices_fully_masked` : indices des dates **entièrement** masquées

Les indices sont relatifs à la fenêtre `t_sampled` (0 à T-1).

In [ ]:
np.random.seed(42)
n_mask = max(1, int(T * 0.4))  # 40% des dates masquées

# ── random_clouds : nuageux aléatoires sur des dates dispersées ──────────
indices_rc = sorted(np.random.choice(T, size=n_mask, replace=False).tolist())
t_masked_rc = {
    "indices_masked": np.array(indices_rc),
    "indices_fully_masked": np.array([], dtype=int),
}

# ── random_fully_masked : dates entières masquées aléatoirement ─────────
indices_rfm = sorted(np.random.choice(T, size=n_mask, replace=False).tolist())
t_masked_rfm = {
    "indices_masked": np.array(indices_rfm),
    "indices_fully_masked": np.array(indices_rfm),
}

# ── consecutive_fully_masked : bloc consécutif entièrement masqué ────────
start_cfm = np.random.randint(0, T - n_mask)
indices_cfm = list(range(start_cfm, start_cfm + n_mask))
t_masked_cfm = {
    "indices_masked": np.array(indices_cfm),
    "indices_fully_masked": np.array(indices_cfm),
}

print(f"T = {T} dates, {n_mask} dates masquées ({n_mask/T:.0%})")
print(f"\n  random_clouds         : indices = {indices_rc}")
print(f"  random_fully_masked   : indices = {indices_rfm}")
print(f"  consecutive_fully     : indices = {indices_cfm} (bloc de {n_mask})")

T = 14 dates, 5 dates masquées (36%)

  random_clouds         : indices = [0, 5, 9, 11, 12]
  random_fully_masked   : indices = [0, 6, 8, 9, 11]
  consecutive_fully     : indices = [5, 6, 7, 8, 9] (bloc de 5)


## 4. Appliquer les 3 masquages et lancer l'inférence

On appelle `test_dset.__getitem__(SAMPLE_IDX, t_sampled=..., t_masked=...)`
pour chaque stratégie, puis on lance l'inférence avec le modèle entraîné.

In [ ]:
def make_mask_config(mask_type):
    return OmegaConf.create({
        "mask_type": mask_type,
        "ratio_masked_frames": 0.5,
        "ratio_fully_masked_frames": 0.4,
        "non_masked_frames": [0],
        "fixed_masking_ratio": False,
        "p_filter": 0.1,
    })

strategies = {
    "random_clouds": {
        "mask_kwargs": make_mask_config("random_clouds"),
        "t_masked": t_masked_rc,
    },
    "random_fully_masked": {
        "mask_kwargs": make_mask_config("random_fully_masked"),
        "t_masked": t_masked_rfm,
    },
    "consecutive_fully_masked": {
        "mask_kwargs": make_mask_config("consecutive_fully_masked"),
        "t_masked": t_masked_cfm,
    },
}

# Sauvegarder l'état courant
saved = {
    "mask_kwargs": test_dset.mask_kwargs,
    "fill_value": test_dset.fill_value,
    "fill_type": test_dset.fill_type,
    "fixed_masking_ratio": test_dset.fixed_masking_ratio,
    "intersect_real_cloud_masks": test_dset.intersect_real_cloud_masks,
    "dilate_cloud_masks": test_dset.dilate_cloud_masks,
}

results = {}
for name, scfg in strategies.items():
    # Injecter la config de masquage
    test_dset.mask_kwargs = scfg["mask_kwargs"]
    test_dset.fill_value = 1.0
    test_dset.fill_type = "fill_value"
    test_dset.fixed_masking_ratio = False
    test_dset.intersect_real_cloud_masks = False
    test_dset.dilate_cloud_masks = False

    sample = test_dset.__getitem__(
        SAMPLE_IDX,
        t_sampled=t_sampled,
        t_masked=scfg["t_masked"],
    )
    batch = {k: v.unsqueeze(0) if isinstance(v, torch.Tensor) else v
             for k, v in sample.items()}

    batch_out, y_pred, _ = imputer.impute_sample(batch, return_att=False)
    inputs, target, masks, mask_valid, cloud_mask, indices_rgb, index_nir = extract_sample(batch_out)
    idx_rgb = indices_rgb.int().tolist()

    results[name] = {
        "inputs": inputs[0],
        "target": target[0],
        "pred": y_pred[0],
        "masks": masks[0],
        "idx_rgb": idx_rgb,
    }
    print(f"  {name:30s} OK  pred={y_pred.shape}")

# Restaurer l'état
for k, v in saved.items():
    setattr(test_dset, k, v)

print("\nInference terminee pour les 3 strategies.")

AttributeError: 'SentinelDataset' object has no attribute 'patches_dataset'

## 5. Visualisation comparative

Pour chaque stratégie : **Cible | Entrée masquée | Prédiction | Masque**

In [ ]:
def plot_strategy(name, res, brightness=3.0, n_show=8):
    idx = res["idx_rgb"]
    target = res["target"][:, idx].permute(0, 2, 3, 1).cpu().clamp(0, 1) * brightness
    inp = res["inputs"][:, idx].permute(0, 2, 3, 1).cpu().clamp(0, 1) * brightness
    pred = res["pred"][:, idx].permute(0, 2, 3, 1).cpu().clamp(0, 1) * brightness
    masks_vis = res["masks"][:, 0].cpu()
    T = target.shape[0]
    n = min(n_show, T)

    fig, axes = plt.subplots(4, n, figsize=(2.5 * n, 9))
    fig.suptitle(f"{name}", fontsize=14, fontweight="bold", y=1.02)

    for j in range(n):
        axes[0, j].imshow(target[j].clip(0, 1)); axes[0, j].axis("off")
        axes[1, j].imshow(inp[j].clip(0, 1));    axes[1, j].axis("off")
        axes[2, j].imshow(pred[j].clip(0, 1));    axes[2, j].axis("off")
        axes[3, j].imshow(masks_vis[j], cmap="gray", vmin=0, vmax=1); axes[3, j].axis("off")
        axes[0, j].set_title(f"t{j}", fontsize=9)

    for row, label in enumerate(["Cible", "Entree masquee", "Prediction", "Masque"]):
        axes[row, 0].annotate(label, xy=(-0.15, 0.5), xycoords="axes fraction",
                              fontsize=10, fontweight="bold", ha="right", va="center", rotation=90)
    plt.tight_layout()
    plt.show()

for name in ["random_clouds", "random_fully_masked", "consecutive_fully_masked"]:
    plot_strategy(name, results[name])

## 6. Comparaison côte à côte des prédictions

Les 3 prédictions sont alignées pour chaque date, avec la cible en référence.

In [ ]:
idx_rgb = results["random_clouds"]["idx_rgb"]
brightness = 3.0
T = results["random_clouds"]["target"].shape[0]
n = min(8, T)
strat_names = ["random_clouds", "random_fully_masked", "consecutive_fully_masked"]
strat_short = ["RC", "RFM", "CFM"]

fig, axes = plt.subplots(4, n, figsize=(2.5 * n, 10))
fig.suptitle("Comparaison des 3 strategies de masquage", fontsize=14, fontweight="bold", y=1.02)

# Ligne 0 : Cible
target_vis = results["random_clouds"]["target"][:, idx_rgb].permute(0, 2, 3, 1).cpu().clamp(0, 1) * brightness
for j in range(n):
    axes[0, j].imshow(target_vis[j].clip(0, 1))
    axes[0, j].axis("off")
    axes[0, j].set_title(f"t{j}", fontsize=9)
axes[0, 0].annotate("Cible", xy=(-0.15, 0.5), xycoords="axes fraction",
                     fontsize=10, fontweight="bold", ha="right", va="center", rotation=90)

# Lignes 1-3 : Prédictions
for i, (name, short) in enumerate(zip(strat_names, strat_short)):
    pred_vis = results[name]["pred"][:, idx_rgb].permute(0, 2, 3, 1).cpu().clamp(0, 1) * brightness
    for j in range(n):
        axes[i + 1, j].imshow(pred_vis[j].clip(0, 1))
        axes[i + 1, j].axis("off")
    axes[i + 1, 0].annotate(f"Pred {short}", xy=(-0.15, 0.5), xycoords="axes fraction",
                            fontsize=10, fontweight="bold", ha="right", va="center", rotation=90)

plt.tight_layout()
plt.show()

## 7. Métriques de reconstruction par stratégie

In [ ]:
from src.metrics.cloud_removal import CloudRemovalMetrics

compute_metrics = CloudRemovalMetrics(
    metrics=["mae", "rmse", "psnr", "ssim", "sam"],
    eval_occluded_observed=True,
)

print(f"{'Strategie':30s} | {'MAE':>8s} | {'RMSE':>8s} | {'PSNR':>8s} | {'SSIM':>8s} | {'SAM':>8s}")
print("-" * 90)

for name in strat_names:
    r = results[name]
    m = compute_metrics(r["target"][0:1], r["masks"][0:1], r["pred"][0:1], None)
    vals = []
    for k in ["mae", "rmse", "psnr", "ssim", "sam"]:
        v = m[k]
        vals.append(f"{v.item():.4f}" if hasattr(v, 'item') else f"{v:.4f}")
    print(f"{name:30s} | {vals[0]:>8s} | {vals[1]:>8s} | {vals[2]:>8s} | {vals[3]:>8s} | {vals[4]:>8s}")

## 8. Résumé

| Stratégie | Description | Usage typique |
|---|---|---|
| `random_clouds` | Masques nuageux aléatoires superposés | Entraînement |
| `random_fully_masked` | Dates entières masquées aléatoirement | Évaluation |
| `consecutive_fully_masked` | Bloc de dates consécutives entièrement masquées | Évaluation (pire cas) |

**Pour une évaluation complète** :
```bash
python run_eval.py configs/config_run_eval.yaml
```

**Pour l'inférence à la tuile** :
```bash
python run_inference.py configs/config_run_inference.yaml
```